# 📊 Módulo 08 - Notebook 03: Waterfall Chart P&L

## 💰 Gráficos de cascada para Estados Financieros

**Libro:** Saliendo de lo Pandito  
**Módulo:** 08 - Visualización Plotly Databricks Dashboards  
**Duración estimada:** 60 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** gráficos de cascada (Waterfall Charts)  
✅ **Visualizar** Estados de Resultados (P&L)  
✅ **Representar** flujos acumulativos  
✅ **Personalizar** colores por tipo de movimiento  
✅ **Generar** reportes financieros ejecutivos

---

## 📋 Pre-requisitos

* ✅ Notebooks 08_01 y 08_02 completados
* ✅ Conocimiento de Estados Financieros P&L
* ✅ Familiaridad con Plotly Graph Objects

---

## 📚 Contenido

1. Teoría de Waterfall Charts
2. Anatomía de un P&L
3. Construcción con Graph Objects
4. Personalización de Colores
5. Caso Integrador: P&L Completo

---

## 💡 Por qué importa

**Waterfall Charts son EL gráfico ejecutivo para finanzas:**

* 💰 **P&L:** Visualizar camino de Ventas → EBITDA
* 📊 **Variaciones:** Explicar cambios período a período
* 💵 **Cash Flow:** Mostrar orígenes y aplicaciones

**El gráfico favorito de CFOs y Controllers**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Simular estructura P&L basada en ventas reales
    np.random.seed(42)
    ventas_totales = df_ventas['ventas'].sum()
    
    # Construir estructura P&L
    pl_data = {
        'Concepto': ['Ventas', 'Costo Ventas', 'Margen Bruto', 'Gastos Operativos', 'EBITDA', 'Depreciación', 'EBIT', 'Intereses', 'EBT', 'Impuestos', 'Resultado Neto'],
        'Monto': [
            ventas_totales,
            -ventas_totales * 0.60,  # 60% COGS
            0,  # Calculated
            -ventas_totales * 0.15,  # 15% OpEx
            0,  # Calculated
            -ventas_totales * 0.03,  # 3% Depreciation
            0,  # Calculated
            -ventas_totales * 0.02,  # 2% Interest
            0,  # Calculated
            -ventas_totales * 0.05,  # 5% Tax on EBT
            0   # Calculated
        ],
        'Tipo': ['absoluto', 'relativo', 'total', 'relativo', 'total', 'relativo', 'total', 'relativo', 'total', 'relativo', 'total']
    }
    
    df_pl = pd.DataFrame(pl_data)
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   💰 Ventas totales: ${ventas_totales:,.2f}")
    print(f"   📊 Registros originales: {len(df_ventas):,}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 Estructura P&L simulada:")
    print(f"   • Basada en ventas reales de Los Andes Market")
    print(f"   • Estructura típica de Estado de Resultados")
    print(f"   • Lista para Waterfall Chart")
    
    print(f"\n🎯 Este notebook usará datos REALES para P&L")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_pl = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Waterfall Charts: El Gráfico Ejecutivo

### 💧 ¿Qué es un Waterfall Chart?

Un **Waterfall Chart** (gráfico de cascada) muestra cómo un valor inicial se transforma mediante adiciones y sustracciones hasta llegar a un valor final.

**Visualización:**
```
  ┌──────┐
  │Ventas│ 1,000
  └──┬───┘
     │ -600 (Costo)
  ┌──▼───┐
  │ MB   │ 400
  └──┬───┘
     │ -150 (Gastos)
  ┌──▼───┐
  │EBITDA│ 250
  └──────┘
```

---

### 📊 Anatomía de un P&L (Estado de Resultados)

**Estructura típica:**

```
Ventas                    $1,000,000
- Costo de Ventas         ($600,000)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
= Margen Bruto            $400,000   (40%)
- Gastos Operativos       ($150,000)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
= EBITDA                  $250,000   (25%)
- Depreciación            ($30,000)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
= EBIT                    $220,000
- Intereses               ($20,000)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
= EBT                     $200,000
- Impuestos (25%)         ($50,000)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
= Resultado Neto          $150,000   (15%)
```

---

### 🎨 Tipos de Barras en Waterfall

| Tipo | Descripción | Color típico |
|------|-------------|---------------|
| **Absoluto** | Valor inicial (Ventas) | 🔵 Azul |
| **Relativo +** | Incremento | 🟢 Verde |
| **Relativo -** | Decremento | 🔴 Rojo |
| **Total** | Subtotales (Margen Bruto, EBITDA, etc.) | 🟡 Amarillo/Dorado |

---

### 🛠️ Construcción en Plotly

**Con Graph Objects:**

```python
import plotly.graph_objects as go

fig = go.Figure(go.Waterfall(
    name = "P&L",
    orientation = "v",
    measure = ["absolute", "relative", "total", "relative", "total"],
    x = ["Ventas", "Costo", "Margen Bruto", "Gastos", "EBITDA"],
    y = [1000, -600, 0, -150, 0],
    text = ["+$1,000", "-$600", "$400", "-$150", "$250"],
    textposition = "outside"
))

fig.update_layout(title="Estado de Resultados P&L")
fig.show()
```

**Parámetros clave:**
* `measure`: Tipo de cada barra ("absolute", "relative", "total")
* `y`: Valores (0 para totales calculados automáticamente)
* `text`: Etiquetas personalizadas

---

### 💼 Casos de Uso

1. **P&L mensual/anual**
2. **Análisis de variaciones:** Budget vs Real
3. **Cash Flow:** Orígenes y Aplicaciones
4. **Ventas:** Precio vs Volumen
5. **Rentabilidad por producto**

---

### 🎯 Ventaja sobre gráficos de barras tradicionales

✅ Muestra **flujo acumulativo**  
✅ Visualiza **relaciones causa-efecto**  
✅ Ideal para **presentaciones ejecutivas**

In [0]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("💧 WATERFALL CHARTS PARA P&L")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")
try:
    import plotly
    print(f"Versión de Plotly: {plotly.__version__}")
except:
    print("⚠️  Plotly no instalado. Ejecuta: %pip install plotly")

print("\n🎯 En este notebook aprenderás:")
print("  • go.Waterfall() - Gráficos de cascada")
print("  • measure: 'absolute', 'relative', 'total'")
print("  • Visualización de P&L (Estado de Resultados)")
print("  • Personalización de colores por tipo")

print("\n📖 Métodos clave:")
print("  - go.Figure(go.Waterfall(...))")
print("  - measure=['absolute', 'relative', 'total']")
print("  - y=[valor, cambio, 0, cambio, 0]  # 0 = calculado")
print("  - fig.update_layout(title='P&L')")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# 🎯 OPCIONAL: Calcular métricas P&L desde datos reales

# Descomentar para usar datos reales:
"""
import pandas as pd
import numpy as np

print("💾 Cargando datos reales desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    # Calcular métricas P&L simuladas (puedes ajustar los % según tu negocio)
    ventas_brutas = df_ventas['ventas'].sum()
    costo_ventas = ventas_brutas * 0.60  # 60% del costo
    margen_bruto = ventas_brutas - costo_ventas
    gastos_operativos = ventas_brutas * 0.15  # 15% de gastos
    ebitda = margen_bruto - gastos_operativos
    
    pl_data = {
        'Concepto': ['Ventas Brutas', 'Costo de Ventas', 'Margen Bruto', 
                     'Gastos Operativos', 'EBITDA'],
        'Monto': [ventas_brutas, -costo_ventas, margen_bruto, 
                  -gastos_operativos, ebitda]
    }
    
    df_pl = pd.DataFrame(pl_data)
    
    print(f"✅ Métricas P&L calculadas desde datos reales:")
    print(f"\n📈 Estado de Resultados (período completo):")
    display(df_pl)
    
    print(f"\n💡 Variable disponible: df_pl")
    print("   Usar para crear gráfico Waterfall con Plotly")
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del notebook 08_03

### ✅ Lo que aprendiste

1. **Waterfall Charts (go.Waterfall):**
   - `go.Figure(go.Waterfall(measure=, x=, y=, text=))`
   - Muestra cómo un valor inicial se transforma mediante adiciones y sustracciones
   - El gráfico ejecutivo por excelencia para finanzas

2. **Parámetro measure (tipo de barra):**
   - `'absolute'`: valor inicial o final (ej: Ventas, Resultado Neto)
   - `'relative'`: incremento o decremento (ej: Costo, Gastos)
   - `'total'`: subtotal calculado automáticamente (ej: Margen Bruto, EBITDA)
   - El orden de measure define cómo se acumula la cascada

3. **Personalización de colores:**
   - `increasing.marker.color` para movimientos positivos (verde)
   - `decreasing.marker.color` para movimientos negativos (rojo)
   - `totals.marker.color` para subtotales (dorado)
   - `connector.line.color` para líneas conectoras entre barras

4. **Anatomía de un P&L:**
   - Ventas → Costo de Ventas → Margen Bruto → Gastos Op → EBITDA
   - EBITDA → Depreciación → EBIT → Intereses → EBT → Impuestos → Neto
   - Cada paso es un `measure` en la cascada

5. **Caso integrador:**
   - P&L completo con datos reales de Los Andes Market
   - Etiquetas con formato monetario (`texttemplate`)
   - Exportar como HTML interactivo para presentaciones ejecutivas

---

### 🎯 Reglas de Oro

👉 **Regla #1: measure define el flujo de la cascada**
```python
# MALO: todos como 'relative', los totales no se calculan
measure = ['relative'] * 5

# BUENO: 'absolute' para inicio, 'relative' para cambios, 'total' para subtotales
measure = ['absolute', 'relative', 'total', 'relative', 'total']
x = ['Ventas', 'Costo', 'Margen Bruto', 'Gastos', 'EBITDA']
y = [1000, -600, 0, -150, 0]  # 0 en totals: se calcula solo
```

👉 **Regla #2: y=0 en los subtotales, Plotly los calcula**
```python
# MALO: poner el valor calculado manualmente
y = [1000, -600, 400, -150, 250]

# BUENO: 0 en los 'total', Plotly acumula automáticamente
y = [1000, -600, 0, -150, 0]
measure = ['absolute', 'relative', 'total', 'relative', 'total']
```

👉 **Regla #3: Formatear etiquetas con texttemplate**
```python
# MALO: sin etiquetas, el gráfico no comunica montos
fig = go.Figure(go.Waterfall(measure=measure, x=x, y=y))

# BUENO: etiquetas con formato monetario
fig = go.Figure(go.Waterfall(
    measure=measure, x=x, y=y,
    text=["+$1,000", "-$600", "$400", "-$150", "$250"],
    textposition="outside"
))
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Estado de Resultados (P&L) | `go.Waterfall` con measure absolute/relative/total |
| Análisis Budget vs Real | `go.Waterfall` con variaciones positivas/negativas |
| Cash Flow (orígenes y aplicaciones) | `go.Waterfall` con relative + y - |
| Variación período a período | `go.Waterfall` partiendo de base absolute |
| Rentabilidad por producto | `go.Waterfall` con margen acumulado |
| Comparación simple entre categorías | `px.bar()` (no waterfall) |
| Evolución temporal | `px.line()` (no waterfall) |
| Colorear positivos/negativos | `increasing.marker.color` / `decreasing.marker.color` |
| Formatear montos en etiquetas | `text=["+$1,000", ...]` + `textposition='outside'` |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>💧 ¡Waterfall Charts y P&L dominados!</h3>
  <p><i>"El Waterfall Chart es el lenguaje visual de los CFOs: muestra el camino de las ventas al resultado neto."</i></p>
</div>